# Predicting corn yield using USDA NASS yield, weather, soil, and possibly satellite data

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Corn Yield Data for the corn belt (Indiana, Illinois, Iowa, etc.)

Data was collected from [United States Department of Agriculture Website](https://www.nass.usda.gov/Quick_Stats/) using their [Quick Stats Tool](https://quickstats.nass.usda.gov/). When looking at their site, you click the categories listed in the table below:

| Field              | Value                                        |
| ------------------ | -------------------------------------------- |
| Program/source     | SURVEY                                       |
| Sector             | CROPS                                        |
| Group              | FIELD CROPS                                  |
| Commodity          | CORN                                         |
| Category           | YIELD                                        |
| Data Item          | CORN, GRAIN - YIELD, MEASURED IN BU / ACRE   |
| Domain             | TOTAL                                        |
| Geographic level   | COUNTY                                       |
| State              | Indiana, Illinois, Iowa                      |
| Ag Distric         | Select All That Are Avalaible                |
| County             | Select All That Are Avalaible                |
| Years              | 2000–2025                                    |
| Period Type        | Annual                                       |
| Period             | YEAR                                         |


I am going to use the [USDA NASS API](https://quickstats.nass.usda.gov/api) to easily download data.

Here are some resources I used to help figure out how to get data from an api in python using the [requests library](https://requests.readthedocs.io/en/latest/):
- [https://dev.to/kiprotichterer/extracting-data-from-an-api-using-python-requests-4h5](https://dev.to/kiprotichterer/extracting-data-from-an-api-using-python-requests-4h5)

In [3]:
import requests

I have an API key that I requested that needs to be used each time I make a request with the api. 

The list of state names will allow me to loop through and get data for each state.

The url is the base url for accessing the API.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

USDA_NASS_API_KEY = os.getenv("USDA_NASS_API_KEY")

state_names = ["ILLINOIS", "INDIANA", "IOWA", "KANSAS", "KENTUCKY", 
               "MICHIGAN", "MINNESOTA", "MISSOURI", "NEBRASKA", 
               "NORTH DAKOTA", "OHIO", "SOUTH DAKOTA", "WISCONSIN"]

url = f"https://quickstats.nass.usda.gov/api/api_GET/"

List of all parameters for API
- [https://quickstats.nass.usda.gov/api#param_define](https://quickstats.nass.usda.gov/api#param_define)

In [5]:
# # See the avaialbe parameter values in the parameter
# parameter = "class_desc"
# url = f"https://quickstats.nass.usda.gov/api/get_param_values/?key={API_KEY}&param={parameter}"
# r = requests.get(url)
# print("*"*30)
# print("status_code")
# print("*"*30)
# print(r.status_code)
# print()
# print("*"*30)
# print("text")
# print("*"*30)
# print(r.text)
# print()
# print("*"*30)
# print("json")
# print("*"*30)
# print(r.json())

In [6]:
all_state_data = []

for state_name in state_names:
    print(f"Downloading data for {state_name}")
    params = {
        "key": USDA_NASS_API_KEY,
        "source_desc":"SURVEY",
        "sector_desc": "CROPS",
        "group_desc": "FIELD CROPS",
        "commodity_desc": "CORN",
        "class_desc": "ALL CLASSES",
        "prodn_practice_desc": "ALL PRODUCTION PRACTICES",
        "util_practice_desc": "GRAIN",
        "statisticcat_desc": "YIELD",
        "unit_desc": "BU / ACRE",
        "agg_level_desc": "COUNTY",
        "state_name": state_name,
        "year__GE": 2000,
        "year__LE": 2025,
        "freq_desc": "ANNUAL",
        "format": "JSON"
    }
    
    r = requests.get(url, params=params)
    data = r.json()
    if "data" not in data:
        raise ValueError(f"No data returned for {state_name}: {data}")
    all_state_data.append(pd.DataFrame(data['data']))

In [7]:
df = pd.concat(all_state_data)

In [8]:
df.head()

,sector_desc,watershed_desc,state_fips_code,util_practice_desc,domaincat_desc,group_desc,country_name,county_code,location_desc,short_desc,...,region_desc,state_name,week_ending,asd_desc,watershed_code,class_desc,county_name,domain_desc,country_code,end_code
0,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"ILLINOIS, NORTHWEST, OTHER (COMBINED) COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,OTHER (COMBINED) COUNTIES,TOTAL,9000,00
1,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"ILLINOIS, NORTHWEST, OTHER (COMBINED) COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,OTHER (COMBINED) COUNTIES,TOTAL,9000,00
2,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00
3,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00
4,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00


## Beginning of exploring data by doing sanity checks

In [9]:
df.shape

(25641, 39)

In [10]:
df.head()

,sector_desc,watershed_desc,state_fips_code,util_practice_desc,domaincat_desc,group_desc,country_name,county_code,location_desc,short_desc,...,region_desc,state_name,week_ending,asd_desc,watershed_code,class_desc,county_name,domain_desc,country_code,end_code
0,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"ILLINOIS, NORTHWEST, OTHER (COMBINED) COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,OTHER (COMBINED) COUNTIES,TOTAL,9000,00
1,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"ILLINOIS, NORTHWEST, OTHER (COMBINED) COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,OTHER (COMBINED) COUNTIES,TOTAL,9000,00
2,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00
3,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00
4,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00


In [11]:
df.tail()

,sector_desc,watershed_desc,state_fips_code,util_practice_desc,domaincat_desc,group_desc,country_name,county_code,location_desc,short_desc,...,region_desc,state_name,week_ending,asd_desc,watershed_code,class_desc,county_name,domain_desc,country_code,end_code
1653,CROPS,,55,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"WISCONSIN, OTHER COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,WISCONSIN,,,00000000,ALL CLASSES,OTHER COUNTIES,TOTAL,9000,00
1654,CROPS,,55,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"WISCONSIN, OTHER COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,WISCONSIN,,,00000000,ALL CLASSES,OTHER COUNTIES,TOTAL,9000,00
1655,CROPS,,55,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"WISCONSIN, OTHER COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,WISCONSIN,,,00000000,ALL CLASSES,OTHER COUNTIES,TOTAL,9000,00
1656,CROPS,,55,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"WISCONSIN, OTHER COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,WISCONSIN,,,00000000,ALL CLASSES,OTHER COUNTIES,TOTAL,9000,00
1657,CROPS,,55,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"WISCONSIN, OTHER COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,WISCONSIN,,,00000000,ALL CLASSES,OTHER COUNTIES,TOTAL,9000,00


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25641 entries, 0 to 1657
Data columns (total 39 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   sector_desc            25641 non-null  object
 1   watershed_desc         25641 non-null  object
 2   state_fips_code        25641 non-null  object
 3   util_practice_desc     25641 non-null  object
 4   domaincat_desc         25641 non-null  object
 5   group_desc             25641 non-null  object
 6   country_name           25641 non-null  object
 7   county_code            25641 non-null  object
 8   location_desc          25641 non-null  object
 9   short_desc             25641 non-null  object
 10  state_ansi             25641 non-null  object
 11  agg_level_desc         25641 non-null  object
 12  congr_district_code    25641 non-null  object
 13  state_alpha            25641 non-null  object
 14  prodn_practice_desc    25641 non-null  object
 15  zip_5                  25

In [13]:
# Finding missing values
df.isnull().sum()

sector_desc              0
watershed_desc           0
state_fips_code          0
util_practice_desc       0
domaincat_desc           0
group_desc               0
country_name             0
county_code              0
location_desc            0
short_desc               0
state_ansi               0
agg_level_desc           0
congr_district_code      0
state_alpha              0
prodn_practice_desc      0
zip_5                    0
reference_period_desc    0
asd_code                 0
CV (%)                   0
source_desc              0
year                     0
county_ansi              0
Value                    0
unit_desc                0
freq_desc                0
commodity_desc           0
begin_code               0
statisticcat_desc        0
load_time                0
region_desc              0
state_name               0
week_ending              0
asd_desc                 0
watershed_code           0
class_desc               0
county_name              0
domain_desc              0
c

In [14]:
df.isnull().sum()/df.shape[0]*100 # percentage of missing values

sector_desc              0.0
watershed_desc           0.0
state_fips_code          0.0
util_practice_desc       0.0
domaincat_desc           0.0
group_desc               0.0
country_name             0.0
county_code              0.0
location_desc            0.0
short_desc               0.0
state_ansi               0.0
agg_level_desc           0.0
congr_district_code      0.0
state_alpha              0.0
prodn_practice_desc      0.0
zip_5                    0.0
reference_period_desc    0.0
asd_code                 0.0
CV (%)                   0.0
source_desc              0.0
year                     0.0
county_ansi              0.0
Value                    0.0
unit_desc                0.0
freq_desc                0.0
commodity_desc           0.0
begin_code               0.0
statisticcat_desc        0.0
load_time                0.0
region_desc              0.0
state_name               0.0
week_ending              0.0
asd_desc                 0.0
watershed_code           0.0
class_desc    

In [15]:
df.duplicated().sum() # duplicates?

0

In [16]:
# Identifying garbage values
# we want to see if there are any characters in the object columns that shouldn't be there
# like a ! or * or $ or something similar
for i in df.select_dtypes(include='object').columns:
    print(df[i].value_counts())
    print("*"*30)

sector_desc
CROPS    25641
Name: count, dtype: int64
******************************
watershed_desc
    25641
Name: count, dtype: int64
******************************
state_fips_code
19    2486
17    2467
20    2208
18    2207
21    2181
39    2117
31    2116
29    2096
27    1929
55    1658
26    1581
46    1441
38    1154
Name: count, dtype: int64
******************************
util_practice_desc
GRAIN    25641
Name: count, dtype: int64
******************************
domaincat_desc
NOT SPECIFIED    25641
Name: count, dtype: int64
******************************
group_desc
FIELD CROPS    25641
Name: count, dtype: int64
******************************
country_name
UNITED STATES    25641
Name: count, dtype: int64
******************************
county_code
998    997
015    315
027    315
011    314
099    310
      ... 
186     25
229     22
235     15
102     12
237      3
Name: count, Length: 123, dtype: int64
******************************
location_desc
MICHIGAN, SOUTHEAST, LIVINGSTON  

## Keeping the needed columns

In [17]:
df.head()

,sector_desc,watershed_desc,state_fips_code,util_practice_desc,domaincat_desc,group_desc,country_name,county_code,location_desc,short_desc,...,region_desc,state_name,week_ending,asd_desc,watershed_code,class_desc,county_name,domain_desc,country_code,end_code
0,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"ILLINOIS, NORTHWEST, OTHER (COMBINED) COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,OTHER (COMBINED) COUNTIES,TOTAL,9000,00
1,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,998,"ILLINOIS, NORTHWEST, OTHER (COMBINED) COUNTIES","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,OTHER (COMBINED) COUNTIES,TOTAL,9000,00
2,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00
3,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00
4,CROPS,,17,GRAIN,NOT SPECIFIED,FIELD CROPS,UNITED STATES,011,"ILLINOIS, NORTHWEST, BUREAU","CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",...,,ILLINOIS,,NORTHWEST,00000000,ALL CLASSES,BUREAU,TOTAL,9000,00


In [18]:
cols_to_keep = ["state_name", "state_alpha", "state_ansi", "county_name", "county_ansi", 
                "year", "short_desc", "unit_desc", "statisticcat_desc", "Value"]

In [19]:
df = df[cols_to_keep]

In [20]:
df.head()

,state_name,state_alpha,state_ansi,county_name,county_ansi,year,short_desc,unit_desc,statisticcat_desc,Value
0,ILLINOIS,IL,17,OTHER (COMBINED) COUNTIES,,2019,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,174.3
1,ILLINOIS,IL,17,OTHER (COMBINED) COUNTIES,,2012,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,157.3
2,ILLINOIS,IL,17,BUREAU,011,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.9
3,ILLINOIS,IL,17,BUREAU,011,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,240.8
4,ILLINOIS,IL,17,BUREAU,011,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,223.5


### Remove the counties that are not listed like (OTHER (COMBINED) COUNTIES)

In [21]:
df = df[(df["county_name"] != "OTHER (COMBINED) COUNTIES") & (df["county_name"] != "OTHER COUNTIES")]

In [22]:
df.head()

,state_name,state_alpha,state_ansi,county_name,county_ansi,year,short_desc,unit_desc,statisticcat_desc,Value
2,ILLINOIS,IL,17,BUREAU,011,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.9
3,ILLINOIS,IL,17,BUREAU,011,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,240.8
4,ILLINOIS,IL,17,BUREAU,011,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,223.5
5,ILLINOIS,IL,17,BUREAU,011,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.1
6,ILLINOIS,IL,17,BUREAU,011,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,203.5


In [23]:
temp = df[df["county_name"] == "BUREAU"]

In [24]:
temp.count()

state_name           26
state_alpha          26
state_ansi           26
county_name          26
county_ansi          26
year                 26
short_desc           26
unit_desc            26
statisticcat_desc    26
Value                26
dtype: int64

In [25]:
df.shape[0] / 26

947.8461538461538

In [26]:
# use groupby to explore this more

In [27]:
# Things to add
# weather
# Soil
# other things that may affect the quality of the corn
# basically anything that can enrich the dataset
# need to show why bringing in weather and soil data is useful to this dataset already

## Creating Federal Information Processing System (FIPS) Codes for States and Counties

[https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt](https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt)

In [28]:
fips = []
for row in df.iterrows():
    fip = row[1][2] + row[1][4] # 2, 4
    fips.append(fip)

df['fips'] = fips

/var/folders/jb/hwwqzh7n2_z5rhqtvy9fq0y40000gn/T/ipykernel_3853/4268735509.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fip = row[1][2] + row[1][4] # 2, 4


In [29]:
df

,state_name,state_alpha,state_ansi,county_name,county_ansi,year,short_desc,unit_desc,statisticcat_desc,Value,fips
2,ILLINOIS,IL,17,BUREAU,011,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.9,17011
3,ILLINOIS,IL,17,BUREAU,011,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,240.8,17011
4,ILLINOIS,IL,17,BUREAU,011,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,223.5,17011
5,ILLINOIS,IL,17,BUREAU,011,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.1,17011
6,ILLINOIS,IL,17,BUREAU,011,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,203.5,17011
...,...,...,...,...,...,...,...,...,...,...,...
1647,WISCONSIN,WI,55,WAUKESHA,133,2004,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,132,55133
1648,WISCONSIN,WI,55,WAUKESHA,133,2003,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,120,55133
1649,WISCONSIN,WI,55,WAUKESHA,133,2002,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,117,55133
1650,WISCONSIN,WI,55,WAUKESHA,133,2001,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,114,55133


### Organizing Columns

In [30]:
df = df[['state_name', 'state_alpha', 'state_ansi', 'county_ansi', 'fips', 'county_name', 'year', 'short_desc',	'unit_desc', 'statisticcat_desc', 'Value']]
df.head()

,state_name,state_alpha,state_ansi,county_ansi,fips,county_name,year,short_desc,unit_desc,statisticcat_desc,Value
2,ILLINOIS,IL,17,011,17011,BUREAU,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.9
3,ILLINOIS,IL,17,011,17011,BUREAU,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,240.8
4,ILLINOIS,IL,17,011,17011,BUREAU,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,223.5
5,ILLINOIS,IL,17,011,17011,BUREAU,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.1
6,ILLINOIS,IL,17,011,17011,BUREAU,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,203.5


## Convert to CSV File

In [31]:
df.to_csv('corn_belt_yield.csv', index=False)